# 14 - Reranker Score-Distribution / Calibration Check

Investigation Task 4. Loads the raw scored-block data from notebook 13
(`data_cache/rerank_scored_blocks_31q.json`) -- no repeat Bedrock Rerank calls.

**Question:** does `relevanceScore` show a clean separation between blocks that contain
gold evidence and blocks that don't (a "cliff"), or is it a smooth continuum? Cohere's own
rerank best-practices docs warn scores are not well-calibrated or comparable across
queries -- this checks whether that shows up empirically on this corpus, which would
argue against ever relying on a fixed `min_score` threshold here (per
`guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md` Sec 1.6's recommendation to
expect `min_score` to sit at 0.0 permanently).

In [1]:
import sys
import json
from pathlib import Path

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

SCORED_BLOCKS_PATH = MODEL_ROOT / "finrag_ml_tg1" / "data_cache" / "rerank_scored_blocks_31q.json"
per_question = json.loads(SCORED_BLOCKS_PATH.read_text())
print(f"Loaded scored blocks for {len(per_question)} questions")

import polars as pl

rows = []
for q in per_question:
    for b in q["blocks"]:
        rows.append({
            "question_id": q["question_id"],
            "final_score": b["final_score"],
            "base_score": b["base_score"],
            "n_sentences": b["n_sentences"],
            "is_gold_block": b["is_gold_block"],
        })

df = pl.DataFrame(rows)
print(f"Total scored blocks: {len(df)}  (gold-containing: {df['is_gold_block'].sum()})")

Loaded scored blocks for 31 questions
Total scored blocks: 910  (gold-containing: 27)


In [2]:
gold_scores = df.filter(pl.col("is_gold_block"))["final_score"]
non_gold_scores = df.filter(~pl.col("is_gold_block"))["final_score"]

def pct(series, p):
    return series.quantile(p, interpolation="linear")

print("=== Score distribution: gold-containing blocks ===")
print(f"  n={len(gold_scores)}  mean={gold_scores.mean():.4f}  median={gold_scores.median():.4f}")
print(f"  p10={pct(gold_scores,0.10):.4f}  p25={pct(gold_scores,0.25):.4f}  "
      f"p75={pct(gold_scores,0.75):.4f}  p90={pct(gold_scores,0.90):.4f}")

print("\n=== Score distribution: non-gold blocks ===")
print(f"  n={len(non_gold_scores)}  mean={non_gold_scores.mean():.4f}  median={non_gold_scores.median():.4f}")
print(f"  p10={pct(non_gold_scores,0.10):.4f}  p25={pct(non_gold_scores,0.25):.4f}  "
      f"p75={pct(non_gold_scores,0.75):.4f}  p90={pct(non_gold_scores,0.90):.4f}")

# Overlap: what fraction of non-gold blocks score AT OR ABOVE the median gold-block score?
# (a large overlap = no clean cliff = fixed threshold is not well-justified here)
gold_median = gold_scores.median()
overlap_frac = (non_gold_scores >= gold_median).mean()
print(f"\nFraction of NON-gold blocks scoring >= the median GOLD-block score: {overlap_frac:.3f}")
print("(near 0.5 = scores barely separate gold from non-gold at all; near 0.0 = clean separation)")

=== Score distribution: gold-containing blocks ===
  n=27  mean=0.5175  median=0.5181
  p10=0.0906  p25=0.3306  p75=0.7723  p90=0.8738

=== Score distribution: non-gold blocks ===
  n=883  mean=0.2363  median=0.1579
  p10=0.0292  p25=0.0672  p75=0.3714  p90=0.5508

Fraction of NON-gold blocks scoring >= the median GOLD-block score: 0.120
(near 0.5 = scores barely separate gold from non-gold at all; near 0.0 = clean separation)


In [3]:
# Threshold sweep: for a range of min_score cutoffs, what fraction of gold blocks would
# survive (recall) vs what fraction of ALL blocks would survive (the "budget" a threshold
# costs you)? This is the empirical case for or against ever using rerank_min_score > 0.
import numpy as np

thresholds = np.round(np.arange(0.0, 1.01, 0.05), 2)
rows = []
for t in thresholds:
    gold_recall = (gold_scores >= t).mean() if len(gold_scores) else None
    all_survival = (df["final_score"] >= t).mean()
    rows.append({"min_score": float(t), "gold_block_recall": gold_recall, "all_block_survival": all_survival})

threshold_df = pl.DataFrame(rows)
print(threshold_df)

shape: (21, 3)
┌───────────┬───────────────────┬────────────────────┐
│ min_score ┆ gold_block_recall ┆ all_block_survival │
│ ---       ┆ ---               ┆ ---                │
│ f64       ┆ f64               ┆ f64                │
╞═══════════╪═══════════════════╪════════════════════╡
│ 0.0       ┆ 1.0               ┆ 1.0                │
│ 0.05      ┆ 0.925926          ┆ 0.817582           │
│ 0.1       ┆ 0.888889          ┆ 0.641758           │
│ 0.15      ┆ 0.814815          ┆ 0.524176           │
│ 0.2       ┆ 0.814815          ┆ 0.448352           │
│ …         ┆ …                 ┆ …                  │
│ 0.8       ┆ 0.222222          ┆ 0.025275           │
│ 0.85      ┆ 0.148148          ┆ 0.008791           │
│ 0.9       ┆ 0.037037          ┆ 0.002198           │
│ 0.95      ┆ 0.0               ┆ 0.0                │
│ 1.0       ┆ 0.0               ┆ 0.0                │
└───────────┴───────────────────┴────────────────────┘


## Interpretation (fill in after running)

- If the gold-vs-non-gold score distributions overlap heavily (overlap fraction well above
  ~0.2-0.3), that's empirical confirmation of Cohere's own calibration warning on this
  corpus -- any fixed `min_score` threshold would either cut real evidence or admit a lot of
  noise, and top-N is the more defensible mechanism (matches the ANALYSIS doc's existing
  recommendation, now with corpus-specific evidence rather than a vendor-docs citation alone).
- If there IS a clear separation (gold blocks reliably score well above non-gold), that's a
  more optimistic read on the cross-encoder's discrimination than Gate 0's 48% pass rate
  suggested (notebook 12) -- worth reconciling the two findings rather than picking whichever
  is more convenient.
- The threshold sweep table shows concretely what any given `min_score` would cost in gold
  recall vs. how many blocks it actually removes -- useful even if the final decision is to
  keep `min_score: 0.0` permanently, since it documents *why* that's the right call rather
  than leaving it unexamined.